In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Mon Feb 22 20:09:58 2021

@author: ronguy
"""

import numpy as np
import matplotlib

import matplotlib.pyplot as plt

import time
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.cluster import KMeans
import umap
from sklearn.cluster import DBSCAN
from sklearn import metrics

from tqdm import tqdm_notebook
from lmfit import minimize, Parameters
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as pl
import shap

import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
%load_ext autoreload
%autoreload 2

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
Run="CyTOF9_Hallel"

import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/Code/")
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/BRCA_SHAP/")
%load_ext autoreload
%autoreload 2
from CyTOFHelper import *

#from SHAPset import *
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
import xgboost as xgb
import umap
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import pandas as pd
from cytof_transform import *

# Load and initialize

In [ ]:
import glob

In [ ]:
dir="/Users/ronguy/Dropbox/CyTOF_Breast/CyTOF_CR7/CyTOF7_scMCF7/csv_scale_value/"

In [ ]:
FList=glob.glob(dir+"*")

In [ ]:
FList.sort()
FList

In [ ]:
DBs=[f"exp{i:02}" for i in range(1,7)]

In [ ]:
for F,DB in zip(FList,DBs):
    print(F)
    globals()[DB]=pd.read_csv(F,)

In [ ]:
Rep=dict(zip(list(exp01.columns),[f.split("_")[-1] for f in list(exp01.columns)]))

In [ ]:
Rep

In [ ]:
for DB in DBs:
    globals()[DB].rename(columns=Rep,inplace=True)

In [ ]:
#dir="/Users/ronguy/Dropbox/WIS-CIMA colab - Analysis/#3 CyTOF  - KPC sample, after CD45 depletion/"

params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")


In [ ]:
Rep=dict(pd.read_excel("/Users/ronguy/Dropbox/Work/CyTOF/Mapping.xlsx").iloc[:,:].values)

In [ ]:
Rep['H3K27Ac']='H3K27ac'

In [ ]:
for F,DB in zip(FList,DBs):
    globals()[DB].rename(columns=Rep,inplace=True)


In [ ]:
N=list(exp01.columns)
N.sort()


In [ ]:
N

In [ ]:
NamesAll,EpiCols,NormMRK,CellIden,CellCyle=GetMarkers(N)

In [ ]:
EpiCols.remove('BMI1')
CellIden.append('BMI1')

In [ ]:
EpiCols.remove('EZH2')
CellIden.append('EZH2')

In [ ]:
%matplotlib inline
hKWD={'element':'step','fill':False,'stat':'density'}

In [ ]:
for DB in DBs:
    plt.figure()
    D=np.arcsinh(globals()[DB]/5).copy()
    sns.histplot(data=D,x='H3',**hKWD,color='blue')
    sns.histplot(data=D,x='H3.3',**hKWD,color='red')
    sns.histplot(data=D,x='H4',**hKWD,color='magenta')
#    sns.histplot(data=D,x='H2A',**hKWD,color='g')
    plt.title(DB)
#plt.xscale('log')
#plt.yscale('log')

## Gate on H3.3/H2A too low, but also remove outliers 99.99% from all 

In [ ]:
GateColumns=['H3.3','H4','H3']



def Gate(data,name):
    ddf=data.copy()
    print(name)
    print("Initial ",len(ddf))
    ddf=ddf[(ddf[GateColumns]>5).all(axis=1)]
    print("Core Gate ",len(ddf))
#    ddf=ddf[(ddf<np.quantile(ddf,0.9999,axis=0)).all(axis=1)]
    print("Outlier Gate ",len(ddf))
    data=ddf.copy()
    del ddf
    return data




In [ ]:
for DB in DBs:
    globals()[DB]=Gate(globals()[DB],DB)



In [ ]:
scFac=5
for DB in DBs:
    globals()[DB]=np.arcsinh(globals()[DB]/scFac)


In [ ]:
for DB in DBs:
    globals()[DB]['Line']=DB

In [ ]:
CAll=pd.concat([globals()[DB].sample(5000,replace=False) for DB in DBs])

In [ ]:
CAll.reset_index(inplace=True)
CAll.to_csv("~/Temp/Test.csv",index=False)

In [ ]:
control_markers = ["H3.3", "H3", "H4", ]
markers_to_correct = NormMRK

In [ ]:
tech1, loadings, ev = plot_tech_factor_qc(
    asinh_data=CAll,
    control_markers=["H3.3", "H3", "H4"],
    tech_factor=None,    # or result_global.tech_factor
    tech_name="tech1",
)

In [ ]:
CAll.reset_index(inplace=True,drop=True)

In [ ]:
cfg = CytofTransformConfig(
    control_markers=control_markers,
    markers_to_correct=markers_to_correct,
    use_compartments=False,
    n_pcs_for_T=1,
    anchor_to_median=True,
    zscore=False,
    line_col='Line'
)
asinh_data=CAll.copy()
result_global = cytof_transform_global(asinh_data, cfg)
asinh_corr_global = result_global.corrected.copy()
zres_global = result_global.residuals_z.copy()

In [ ]:
corr_df = plot_marker_correlations_qc(
    asinh_pre=CAll,               # unnormalized
    asinh_post=asinh_corr_global,
    tech_factor=result_global.tech_factor,
    #markers_to_highlight=NamesAll,
    #     "H3.3", "H3", "H4",
    #     "H3K27ac", "H3K4me3", "H3K9ac",
    #     "KI67"
    # ] ,
    top_n=35,
)

In [ ]:
marker_groups = {
    "core_histones": ["H3.3", "H3", "H4"],
    "PTMs": ["H3K27ac", "H3K4me3", "H3K9ac", "H3K27me3", "H3K36me3"],
    "proliferation": ["KI67"],
    "epithelial": ["ER", "GATA3", "KRT8-18", "KRT5"],
}

plot_gamma_qc(result_global.gamma, marker_groups=marker_groups)

In [ ]:
for DB in DBs:
    M=asinh_corr_global['Line']==DB
    globals()[DB]=asinh_corr_global[M][NamesAll].copy()

In [ ]:
MRK_All=NamesAll.copy()
MRK_All.remove('H3')
MRK_All.remove('H3.3')
MRK_All.remove('H4')
#MRK_All.remove('H2A')

EPC=EpiCols.copy()
Core=['H3','H3.3','H4']#,'H2A']
for C in Core:
    EPC.remove(C)

In [ ]:
NC=2000
aaaa=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    aaaa=pd.concat([aaaa,globals()[DB].sample(NC,replace=False)]).copy()
                  

                
m=np.mean(aaaa,axis=0)
s=np.std(aaaa,axis=0)

for DB in DBs:
    globals()[DB]=(globals()[DB]-m)/s
    globals()[DB]['Line']=DB


params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")

In [ ]:
import distinctipy

In [ ]:
DBCLR=dict(zip(DBs,distinctipy.get_colors(6)))

In [ ]:
a=[]
fig, axs = plt.subplots(11, 3, figsize=(20, 20))
for i, row in enumerate(axs):
    for j, ax in enumerate(row):
        a.append(ax)
        
for i,N in enumerate(MRK_All):
    print(N)
    for DB in DBs:
        
        sns.histplot(data=globals()[DB],x=N,ax=a[i],**hKWD,color=DBCLR[DB],label=f'{DB}')

    
#    a[i].set_title(N)

plt.subplots_adjust(wspace=0.5, hspace=1.9)
a[0].legend()    

#fig.savefig("Plots/AllRaw.pdf",dpi=200,bbox_inches='tight')

In [ ]:
pKWD={'dpi':200,'bbox_inches':'tight'}

In [ ]:
DBs

In [ ]:
CAll=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    CAll=pd.concat([CAll,globals()[DB]]).copy()


In [ ]:
Mat=CAll.groupby('Line').mean()

In [ ]:
Mat=Mat.loc[DBs,:]

In [ ]:
plt.figure(figsize=(5,10))
sns.clustermap(np.round(Mat[MRK_All].T,2),annot=True,cmap=plt.cm.seismic,center=0,yticklabels=True,)
plt.xticks(fontsize=12);
plt.yticks(fontsize=12);
#plt.savefig(f'Plots/HM1.png',**pKWD)

In [ ]:
NC=5000
CAll=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    CAll=pd.concat([CAll,globals()[DB].sample(NC,replace=False)]).copy()
                  

In [ ]:
UM=umap.UMAP(min_dist=0.001,n_neighbors=160,random_state=42,verbose=True)

X_2d=UM.fit_transform(CAll[EPC])
plt.scatter(X_2d[:,0],X_2d[:,1],s=1,c='gray')
plt.show()

In [ ]:
AD=ad.AnnData(CAll[MRK_All],obsm={'X_umap':X_2d},obs=CAll[['Line']])

In [ ]:
sc.pl.umap(AD,color=MRK_All+['Line'],cmap='seismic',vmin='p01',vmax='p99',show=False);
#plt.savefig("Plots/UMAP_All.png",dpi=200,bbox_inches='tight')

In [ ]:
marker_sets = {
    # Luminal/epithelial program: epithelial & ER axis up; mesenchymal/basal down
    "Epithelial_Luminal": {
        "up":   {"EpCAM", "E-cadherin", "ER", "GATA3", "KRT8-18"},# "Pan-KRT",
        "down": {"KRT5", "Vimentin", "aSMA"}
    },

    "Basal_Noa": {
        "up":   {"H3K4me1", "H3K4me3","H3K9me2"},
        "down": {"H4K20me3","H3K36me3"}
    },

    "Proliferation": {
        "up":   {"KI67", "H3S28p", "H3K9ac", "H3K64ac"},
        "down": set()  
    },
}




In [ ]:
Z, P, Zabs, Pabs, Zdir = sipsic_like_scores_v3(
    AD, marker_sets,
    n_perm=2048,                   # or even 16 for smoke test
    prefer_permutation=True,     # <- important
    perm_batch=512,              # keeps RAM flat
    normalize_set_weights="l2",
    use_sparse_W=False,
    progress=True               # progress bars can add overhead in some envs
)

In [ ]:
ADUS=ad.AnnData(obs=Z)
ADUS.obsm['X_umap']=AD.obsm['X_umap']
#ADUS.obs['Class']=AD.obs['Class'].values
sc.pl.umap(ADUS,color=list(ADUS.obs.columns),cmap='magma_r',show=False,vcenter=0,)

In [ ]:
sdf=pd.DataFrame(gaussian_smooth_all_torch(Z.values,AD.obsm['X_umap'],-1),columns=Z.columns)

ADS=ad.AnnData(obs=sdf)
ADS.obsm['X_umap']=AD.obsm['X_umap']
sc.pl.umap(ADS,color=list(ADS.obs.columns),cmap='magma_r',show=False,vcenter=0,);

In [ ]:
AD.write_h5ad("/Users/ronguy/Temp/Test.h5ad")

In [ ]:
fig,ax = plt.subplots(2,3,figsize=(15,10))
a=ax.flatten()


for i,L in enumerate(CAll.Line.unique()):
   
    M=CAll.Line==L
    
    a[i].scatter(X_2d[M,0],X_2d[M,1],s=1,label=f"{L}",cmap=plt.cm.seismic,c=DBCLR[L])

    a[i].legend(markerscale=10,fontsize=10)

fig.savefig(f"Plots/All_Lines.png",bbox_inches='tight',dpi=200)

In [ ]:
import networkx

In [ ]:
from leidenalg import find_partition

In [ ]:
sc.pp.neighbors(AD)

In [ ]:
sc.tl.leiden(AD,resolution=0.4)

In [ ]:
sc.pl.umap(AD,color=['leiden'],cmap='seismic',vmin='p01',vmax='p99',show=False);

In [ ]:
lbl=AD.obs['leiden'].values.astype(int)

In [ ]:
#lbl[lbl==1]=0
#lbl[lbl==2]=0

#lbl[lbl==5]=0

In [ ]:
AD.obs['leiden']=lbl
AD.obs['leiden']=AD.obs['leiden'].astype('category')

In [ ]:
sc.pl.umap(AD,color=['leiden'],cmap='seismic',vmin='p01',vmax='p99',show=False);
plt.savefig("Plots/UMAP_All_Cl.png",dpi=200,bbox_inches='tight')

In [ ]:
pd.crosstab(AD.obs['Line'],AD.obs['leiden']).to_csv("Plots/PreClusterPop.csv")

In [ ]:
CAll['Cl']=AD.obs['leiden'].astype(int).values

In [ ]:
Mat=CAll.groupby(['Cl','Line']).mean(numeric_only=True)

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(Mat[EPC].T,cmap='seismic',center=0,annot=True,annot_kws={'size':8},xticklabels=True)
#plt.savefig("Plots/HM_Cl.png",dpi=200,bbox_inches='tight')

In [ ]:
MRK=list(set(NamesAll).difference(set(EpiCols)))

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(Mat[MRK].T,cmap='seismic',center=0,annot=True,annot_kws={'size':8},xticklabels=True)
#plt.savefig("Plots/HM_Cl_2.png",dpi=200,bbox_inches='tight')

In [ ]:
AD.write_h5ad("/Users/ronguy/Temp/Test.h5ad")

In [ ]:
for DB in DBs:
    M=AD.obs['Line']==DB
    sc.pl.umap(AD[M],color=MRK_All,cmap='seismic',vmin='p01',vmax='p99',show=False);
    plt.savefig(f"Plots/{DB}_UMAP_Epi.png",dpi=200,bbox_inches='tight')

In [ ]:
X_2d=UM.fit_transform(CAll[CellIden])
plt.scatter(X_2d[:,0],X_2d[:,1],s=1,c='gray')
plt.show()

In [ ]:
AD=ad.AnnData(CAll[MRK_All],obsm={'X_umap':X_2d},obs=CAll[['Line']])

In [ ]:
sc.pl.umap(AD,color=MRK_All+['Line'],cmap='seismic',vmin='p01',vmax='p99',show=False);
plt.savefig("Plots/UMAP_All_CellIden.png",dpi=200,bbox_inches='tight')

In [ ]:
fig,ax = plt.subplots(2,3,figsize=(15,10))
a=ax.flatten()


for i,L in enumerate(CAll.Line.unique()):
   
    M=CAll.Line==L
    
    a[i].scatter(X_2d[M,0],X_2d[M,1],s=1,label=f"{L}",cmap=plt.cm.seismic,c=DBCLR[L])

    a[i].legend(markerscale=10,fontsize=10)

fig.savefig(f"Plots/All_Lines_CellIden.png",bbox_inches='tight',dpi=200)

In [ ]:
for DB in DBs:
    M=AD.obs['Line']==DB
    sc.pl.umap(AD[M],color=MRK_All,cmap='seismic',vmin='p01',vmax='p99',show=False);
    plt.savefig(f"Plots/{DB}_UMAP_CellIden.png",dpi=200,bbox_inches='tight')
    

In [ ]:
AD